# 08. π-Flow operation: grid head → policies → native sampler

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf).
<!-- study-navigation:end -->

Проследим expanded channels, normalize layouts, schedule, packed/batched policy math, actual NFE и короткий финальный сегмент.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lab 1 Euler + Lab 2 FM — предпосылки. Grid-head, MultimodalDXPolicy, короткий final segment и network NFE ≠ policy substeps изучаем по K6 source / π-Flow paper, не приписываем MIT.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='08_piflow_grid_head_and_native_sampler';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Physical channels и expanded head

Student head предсказывает G наборов параметров на physical coordinate. При video C_v=16, audio C_a=40 и G=10 output widths=160/400; state по-прежнему содержит 16/40 channels.

$$
\mathrm{head}_v:\ D_v\to p_t p_h p_w(GC_v),\qquad
\mathrm{head}_a:\ D_a\to GC_a.
$$

`PiFlowDiffusionTransformer3D` принимает **per-grid** C_v,C_a и сам умножает на G. HF config может уже хранить expanded widths; повторное умножение дало бы G²C. После forward reshape/movedim создаёт B,G,T,H,W,C_v и B,G,A,C_a.

`_normalize_grid_prediction` принимает canonical grid-first, single-output без grid (добавляет G=1) и legacy grid-last. Некорректные shapes отвергает. При одинаковых размерах разных axes одна shape проверка не устанавливает смысл осей; нужны известные index values и metadata.


In [ ]:
model=tiny_model(grid=3).eval();v,a,kw=tiny_inputs(model)
with torch.no_grad():pv,pa=fresh_forward(model,kw)
assert pv.shape==(1,3,2,4,4,4) and pa.shape==(1,3,6,4)
ps=native('core.algo.piflow_sampler')
canonical=torch.arange(1*3*2*4*4*4).reshape(1,3,2,4,4,4)
legacy=canonical.movedim(1,-2)
assert torch.equal(ps._normalize_grid_prediction(legacy,v,'video'),canonical)
assert ps._normalize_grid_prediction(v,v,'video').shape==(1,1,2,4,4,4)
try:ps._normalize_grid_prediction(torch.zeros(1,7,3),a,'audio')
except ValueError:pass
else:raise AssertionError('Invalid grid must be rejected')
metrics['native_grid_shapes']={'video':list(pv.shape),'audio':list(pa.shape),'physical_video_channels':v.shape[-1]}
show_source('core/components/piflow_dit.py');show_source('core/algo/piflow_sampler.py','_normalize_grid_prediction')


## 2. Почему final segment короче

Пусть S network evaluations и final_step_size_scale=r>0. Требуем, чтобы S−1 обычных raw segments длины h и последний rh покрыли unit interval:

$$
(S-1)h+rh=1\quad\Longrightarrow\quad h=\frac{1}{S-1+r}.
$$

Native default r=.5. При S=10 обычный segment≈0.105263, final≈0.052632. Destination clamp eps оставляет маленький ненулевой final time. Если r>1, final segment длиннее; source не ограничивает r сверху. Это schedule parameter, не математическая необходимость .5.

Важно: num_policy_substeps — density **по единичному raw interval**. Каждый segment получает max(1,round(Δτ·density)). Поэтому суммарное число evaluations близко density, но не обязано равняться ему из-за rounding и minimum. Grid points G, network evaluations S и substep density K независимы.


In [ ]:
rows=[]
for S in [1,2,4,10]:
    h=1/(S-1+.5);tau=torch.tensor(1.)
    counts=[];sizes=[]
    for i in range(S):
        size=h*(.5 if i==S-1 else 1.);dst=(tau-size).clamp(min=1e-6)
        n=int(((tau-dst)*128).round().clamp(min=1));counts.append(n);sizes.append(float(tau-dst));tau=dst
    rows.append({'network_NFE':S,'grid_points_example':10,'policy_density':128,'policy_evaluations_per_modality':sum(counts),'per_segment':counts,'raw_lengths':sizes})
metrics['policy_schedule_counts']=rows
assert rows[-1]['per_segment']==[13]*9+[7] and rows[-1]['policy_evaluations_per_modality']==124
print(json.dumps(rows,indent=2))


## 3. MultimodalDXPolicy: независимые policies, совместно предсказанные параметры

Один fused DiT call видит video/audio и выдаёт обе grids. Далее policy_V и policy_A интегрируются отдельно с собственными shapes. Внутри cheap policy substeps заново cross-attention не вычисляется: cross-modal зависимость уже вошла в source prediction grids. На следующем segment она обновится новым fused call.

Batched video имеет B,T,H,W,C; audio — B,A,D. В packed layout first axis=sum_T/sum_A: σ_s и raw interval повторяются по sequence lengths каждого batch item. Это math broadcasting contract, не доказательство, что любой ragged packed attention runtime изолирует clips без подходящих masks.

`detach` multimodal policy возвращает новый wrapper с detached coefficient tensors. Рассмотрим один прямой instance без network.


In [ ]:
pm=native('core.algo.piflow_math')
xs_v=torch.randn(2,3,2,2,4);xs_a=torch.randn(2,5,4);uv=torch.randn(2,3,3,2,2,4,requires_grad=True);ua=torch.randn(2,3,5,4,requires_grad=True)
p=pm.MultimodalDXPolicy(uv,xs_v,ua,xs_a,torch.ones(2),torch.full((2,),.5),torch.ones(2),torch.full((2,),.5))
det=p.detach();assert not det.policy_V.denoising_output_x_0.requires_grad and not det.policy_A.denoising_output_x_0.requires_grad
same=p.detach_();assert same is p and not p.policy_V.denoising_output_x_0.requires_grad
metrics['multimodal_detach']=True


## 4. Полный native sampler с random tiny grid-DiT

Теперь исполним оригинальный sampler. Начальный noise и веса фиксированы, меняется только policy density. Для S=2 hook должен насчитать ровно 2 transformer calls при любой density. Runtime finite/shape проверки не означают осмысленное audio/video из random model.

Native sampler требует оба latent tensors и multimodal transformer. sample_video=False не удаляет video input и не делает audio-only released sampler. CFG у этой функции отсутствует; обычный CFG argument нельзя незаметно добавлять.


In [ ]:
Bundle=native('core.types').LatentBundle
bundle=Bundle(v.clone(),a.clone(),torch.tensor([0,2],dtype=torch.int32),torch.tensor([0,6],dtype=torch.int32))
rows=[]
for density in [8,32,128]:
    calls=[];hook=model.register_forward_hook(lambda *args:calls.append(1));model.clear_text_proj_cache()
    out=ps.piflow_denoise_loop(clone_bundle(bundle),model,{'text_embeds':kw['text_embed'],'pooled_embed':kw['pooled_text_embed']},
      kw['visual_rope'],kw['audio_rope'],kw['text_rope'],2,5.,attention_mask=kw['attention_mask'],num_policy_substeps=density)
    hook.remove()
    assert len(calls)==2 and out.video.shape==v.shape and out.audio.shape==a.shape
    assert torch.isfinite(out.video).all() and torch.isfinite(out.audio).all()
    rows.append({'density':density,'network_NFE':len(calls),'video_norm':float(out.video.norm()),'audio_norm':float(out.audio.norm())})
metrics['native_sampler_counts']=rows
# Source guards execute before any model call.
try:ps.piflow_denoise_loop(Bundle(v,None,None,None),model,dummy_text(),None,None,kw['text_rope'],2,1.)
except ValueError:metrics['missing_modality_rejected']=True
else:raise AssertionError('Native sampler requires both modalities')


## 5. Ошибка compute budget и корректный итог

При S segments стоимость в первом приближении S·cost(DiT)+Σn_i·cost(policy). Это accounting approximation; overhead, output-grid width, память и kernels мешают прямому обещанию speedup. Увеличение cheap substeps не увеличивает NFE, но их стоимость не буквально нулевая.

Policy method family, output layout, schedule и checkpoint должны согласоваться. Переключить обычный velocity checkpoint на π sampler без обучения expanded head нельзя. В частности, количество grid channels не равно количеству дополнительных audio/video frames.


In [ ]:
fig,ax=plt.subplots(figsize=(7,3));ax.bar([str(r['network_NFE']) for r in metrics['policy_schedule_counts']],[r['policy_evaluations_per_modality'] for r in metrics['policy_schedule_counts']]);ax.axhline(128,ls='--',c='black',label='Configured density 128');ax.set(xlabel='Network NFE',ylabel='Actual policy evaluations per modality',title='Native rounding and final-segment schedule');ax.legend();fig.tight_layout();fig.savefig(figure_path(LAB,'substep_accounting'));plt.show()


## Измерение → вывод

Native grid wrapper и sampler сохранили physical shapes; NFE не зависел от policy density. Для S=10,K=128,r=.5 получено 124 cheap evaluations на каждую modality. Multimodal policies detach проверен. Это execution/accounting evidence, не measured production acceleration.


## Самостоятельные опыты

1. Проверьте S=1,r=.5 аналитически: почему segment покрывает весь interval?
2. Измените r и получите per-segment counts, не умножая S×128.
3. Реализуйте index-valued legacy/canonical roundtrip с неравными axis lengths.
4. Повторите native sampler с frozen audio и проверьте неизменность state.
5. Какие два source-level действия отличают сетку output head от latent state?

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
